In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/data
!ls

/content/drive/MyDrive/data
final_submission2.csv  images			submission_template_v3.csv
final_submission6.csv  sample_labels
final_submission.csv   submission_template.csv


In [ ]:
import os
import time
import json
import re
import pandas as pd
import google.generativeai as genai
from collections import defaultdict
from typing import List, Dict, Any

In [ ]:
GEMINI_API_KEY = ""
MODEL_NAME = "gemini-3-flash-preview"
genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
def get_model():
    generation_config = {
        "temperature": 0.0,
        "top_p": 1.0,
        "max_output_tokens": 8192,
        "response_mime_type": "application/json",
        "response_schema": {
            "type": "object",
            "properties": {
                "constituency_id": {"type": "string"},
                "results": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "candidate_number": {"type": "integer"},
                            "candidate_name": {"type": "string"},
                            "party": {"type": "string"},
                            "votes": {"type": "integer"}
                        },
                        "required": ["candidate_number", "candidate_name", "votes"]
                    }
                },
                "total_votes_reported": {"type": "integer"}
            }
        }
    }

    system_instruction = """
    You are a high-precision OCR and data extraction agent specializing in Thai official election documents (ECT Form).
    Your primary goal is 100% accuracy in numerical extraction from handwritten or typed tables.

    Processing Rules:
    1. STRICT NUMERIC CONVERSION: Convert all Thai numerals (๑, ๒, ๓...) to Arabic numerals (1, 2, 3...).
    2. TABLE FOCUS: Extract data from columns: "หมายเลขผู้สมัคร", "ชื่อ-ชื่อสกุล", "พรรคการเมือง", and "คะแนนที่ได้รับ".
    3. MULTI-PAGE MERGE: Consolidate candidates from all provided images into one list.
    4. HANDWRITING RECOGNITION: Be extremely careful with handwritten numbers.
    5. CLEANING: Remove commas from numbers before converting to integers.
    6. Return ONLY a valid JSON object.
    """

    return genai.GenerativeModel(
        model_name=MODEL_NAME,
        generation_config=generation_config,
        system_instruction=system_instruction
    )

In [ ]:
def get_election_data(image_paths: List[str], constituency_id: str) -> Dict[str, Any]:
    model = get_model()
    prompt_text = (
        f"Analyze these election result pages for {constituency_id}. "
        "Extract every candidate's name, number, and their total votes. "
        "Verify digits carefully, especially handwritten ones."
    )

    content = [prompt_text]
    for path in image_paths:
        if os.path.exists(path):
            with open(path, "rb") as f:
                image_data = f.read()
                mime_type = "image/png" if path.lower().endswith((".png", ".jpg")) else "image/png"
                content.append({"mime_type": mime_type, "data": image_data})

    for i in range(5):
        try:
            response = model.generate_content(content)
            return json.loads(response.text)
        except Exception as e:
            wait_time = 2 ** i
            print(f"Error for {constituency_id}: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)

    return {"constituency_id": constituency_id, "results": []}

In [ ]:
def get_grouped_files(folder_path):
    groups = defaultdict(list)
    if not os.path.exists(folder_path):
        print(f"Warning: Folder {folder_path} not found.")
        return {}

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    for filename in files:
        match = re.match(r"(constituency_\d+_\d+|party_list_\d+_\d+)(_page\d+)?\.(png|jpg|jpeg)", filename, re.IGNORECASE)
        if match:
            group_id = match.group(1)
            groups[group_id].append(os.path.join(folder_path, filename))

    for g in groups:
        groups[g].sort()
    return groups

In [ ]:
def map_to_template(ocr_data, template_csv_path, output_path, target_doc_id):

    new_votes_list = [res['votes'] for res in ocr_data.get('results', [])]
    votes_iterator = iter(new_votes_list)

    current_file = output_path if os.path.exists(output_path) else template_csv_path
    if not os.path.exists(current_file):
        print(f"Error: Template file {template_csv_path} not found.")
        return None

    df = pd.read_csv(current_file)

    if 'doc_id' not in df.columns or 'votes' not in df.columns:
        print("Error: CSV must contain 'doc_id' and 'votes' columns.")
        return df

    def update_votes(row):
        if str(row['doc_id']).strip() == target_doc_id:
            try:
                return next(votes_iterator)
            except StopIteration:
                return row['votes']
        return row['votes']

    # Update Votes
    df['votes'] = df.apply(update_votes, axis=1)

    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    return df

In [ ]:
def process_all(image_dir, template_csv, output_csv):
    grouped_images = get_grouped_files(image_dir)

    if not grouped_images:
        print("No valid images found in the directory.")
        return

    for cid, file_list in grouped_images.items():
        print(f"--- Processing {cid} ({len(file_list)} pages) ---")

        # Gemini
        ocr_result = get_election_data(file_list, cid)

        # Mapping on CSV
        print(f"Mapping {len(ocr_result.get('results', []))} candidates to template...")
        map_to_template(ocr_result, template_csv, output_csv, cid)

        print(f"Finished {cid}\n")

In [ ]:
if __name__ == "__main__":
    IMAGE_DIR = 'images/'
    TEMPLATE_CSV = 'submission_template_v3.csv'
    OUTPUT_CSV = 'test_test.csv'

    process_all(IMAGE_DIR, TEMPLATE_CSV, OUTPUT_CSV)
    print(f"การประมวลผลทั้งหมดเสร็จสิ้น! ผลลัพธ์รวมอยู่ที่: {OUTPUT_CSV}")